# LAB 3 — End-to-End Pipeline: เตรียมข้อมูล → ปรับ Hyperparameter → สร้างโมเดลทำนาย → ทดลองและประเมินผล → GUI

ต่อยอดจาก LAB 2 (MovieLens-100K + Surprise) ให้เป็น **Pipeline ครบวงจร** ตามแนวปฏิบัติในหัวข้อ 5.16

```
Raw ratings ─► [1] Clean ─► [2] k-core ─► [3] Re-index ─► [4] Train / Validation / Test (ตามเวลา) ─► [5] Leakage check
            ─► [6] Default models (Validation) ─► [7] Tuning: GridSearchCV · RandomizedSearchCV · Hold-out (RMSE vs nDCG)
            ─► [8] Refit บน Train+Validation ─► [9] ประเมินครั้งเดียวบน Test ─► [10] Prediction Pipeline (+ re-rank, cold start, save/load) ─► GUI
```

| Part | เนื้อหา |
|---|---|
| 0 | Environment + โหลด MovieLens-100K |
| 1 | Data Preparation Pipeline + Data Card |
| 2 | Core Functions (Model wrapper + Metrics + Evaluation engine — ชุดเดียวกับ LAB 2) |
| 3 | ระบบฐานก่อนปรับจูน (Default hyperparameters) บน Validation |
| 4 | การปรับ Hyperparameter: 4.1 GridSearchCV (SVD) · 4.2 RandomizedSearchCV (KNNBaseline) · 4.3 Hold-out tuning เพื่อ Ranking · 4.4 Overfitting curve |
| 5 | Refit บน Train+Validation แล้วประเมินบน Test: Accuracy + Beyond-Accuracy (Default vs Tuned) |
| 6 | Prediction Pipeline: แนะนำ + Re-ranking ความหลากหลาย (MMR) + Cold-start fallback + Save/Load |
| 7 | ตัวอย่าง Output บน Validation/Test: เทียบ Rating และ Ranking ทุกโมเดล |
| 8 | ตารางสรุป + Chart |
| 9 | GUI (Gradio): ผู้ใช้เดิม · เทียบโมเดล · อธิบายค่าทำนาย · ผู้ใช้ใหม่ (Cold start) |

> เวลารันโดยประมาณบน Colab (CPU): 10–15 นาที
>
> ถ้า `from surprise import ...` แจ้ง error เกี่ยวกับ numpy (binary incompatibility) ให้รัน
> `!pip install -q --no-cache-dir --force-reinstall scikit-surprise` แล้วเลือก Runtime → Restart session ก่อนรันใหม่

In [ ]:
# ============================================================
# Part 0 — Environment & Dataset
# ============================================================
!pip -q install scikit-surprise gradio

import math, random, warnings, itertools, time, re, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import roc_auc_score

from surprise import (Dataset, Reader, accuracy, NormalPredictor, BaselineOnly, KNNBasic, KNNWithMeans,
                      KNNWithZScore, KNNBaseline, SVD, SVDpp, NMF, SlopeOne, CoClustering)
from surprise.model_selection import GridSearchCV, RandomizedSearchCV, cross_validate

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
SEED = 42
np.random.seed(SEED); random.seed(SEED)

GENRES = ["unknown", "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime", "Documentary", "Drama",
          "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"]

def load_movielens_100k():
    """โหลด MovieLens-100K ผ่าน Surprise (ดาวน์โหลดจาก GroupLens ครั้งแรก)
    แล้วอ่าน u.item (ชื่อ/ปี/แนว) และ u.user (อายุ/เพศ/อาชีพ) จากโฟลเดอร์เดียวกัน
    ถ้าดาวน์โหลดไม่ได้ จะใช้สำเนาจาก GitHub (RecBole) เป็นทางสำรอง"""
    base = Path.home() / ".surprise_data" / "ml-100k" / "ml-100k"
    try:
        Dataset.load_builtin("ml-100k", prompt=False)
    except Exception as e:
        print("load_builtin ไม่สำเร็จ:", type(e).__name__)
    if (base / "u.data").exists():
        ratings = pd.read_csv(base / "u.data", sep="\t", names=["user_id", "item_id", "rating", "timestamp"])
        it = pd.read_csv(base / "u.item", sep="|", header=None, encoding="latin-1")
        items = pd.DataFrame({"item_id": it[0], "title": it[1]})
        flags = it.iloc[:, 5:24].values
        items["genre"] = ["|".join(g for g, f in zip(GENRES, row) if f == 1) or "unknown" for row in flags]
        users = pd.read_csv(base / "u.user", sep="|", names=["user_id", "age", "gender", "occupation", "zip"])
        src = str(base)
    else:
        url = "https://raw.githubusercontent.com/RUCAIBox/RecBole/master/dataset/ml-100k/"
        ratings = pd.read_csv(url + "ml-100k.inter", sep="\t"); ratings.columns = ["user_id", "item_id", "rating", "timestamp"]
        it = pd.read_csv(url + "ml-100k.item", sep="\t", dtype=str); it.columns = ["item_id", "title", "year", "genre"]
        it["title"] = it["title"] + " (" + it["year"].fillna("") + ")"
        items = it[["item_id", "title"]].assign(genre=it["genre"].fillna("unknown").str.replace(" ", "|"))
        users = pd.read_csv(url + "ml-100k.user", sep="\t"); users.columns = ["user_id", "age", "gender", "occupation", "zip"]
        src = url
    for df in (ratings, items, users):
        for c in ("user_id", "item_id"):
            if c in df.columns: df[c] = df[c].astype(str)
    ratings["rating"] = ratings["rating"].astype(float)
    items["year"] = items["title"].str.extract(r"\((\d{4})\)\s*$")[0]
    return ratings, items, users, src

ratings, items, users, DATA_SRC = load_movielens_100k()
TITLE = dict(zip(items.item_id, items.title)); GENRE = dict(zip(items.item_id, items.genre))
YEAR = dict(zip(items.item_id, items.year))
READER = Reader(rating_scale=(1, 5))
LIKE_T = 4.0
print("Source:", DATA_SRC, "| raw ratings:", len(ratings))

## Part 1 — Data Preparation Pipeline (หัวข้อ 5.16)
แต่ละขั้นเป็นฟังก์ชันที่รับ DataFrame และคืน DataFrame พร้อมบันทึกลง **Data Card** เพื่อให้ทำซ้ำได้
1. **Clean**: ตัดค่าว่าง ตรวจช่วงคะแนน 1–5 และกำจัดคะแนนซ้ำ (เก็บค่าล่าสุด)
2. **k-core**: ตัดผู้ใช้/รายการที่มีคะแนนน้อยกว่า k ซ้ำจนคงที่ (ต้องรายงานค่า k เสมอ)
3. **Re-index**: แปลงรหัสเป็น 0..|U|-1, 0..|I|-1 และสร้างเมทริกซ์เบาบาง (CSR)
4. **Split ตามเวลา**: ของผู้ใช้แต่ละคน 80% แรก → Train, 10% ถัดไป → Validation, 10% ล่าสุด → Test
5. **Leakage check**: ทุกคะแนนใน Validation/Test ต้องเกิดหลังคะแนนใน Train ของผู้ใช้คนเดียวกัน

In [ ]:
from scipy.sparse import csr_matrix
DATA_CARD = []
def log_step(step, df, note=""):
    nu, ni = df.user_id.nunique(), df.item_id.nunique()
    DATA_CARD.append({"step": step, "ratings": len(df), "users": nu, "items": ni,
                      "density": len(df) / (nu * ni) if nu and ni else 0, "note": note})
    return df

def step_clean(df):
    d = df.dropna(subset=["user_id", "item_id", "rating"])
    d = d[(d.rating >= 1) & (d.rating <= 5)]
    d = d.sort_values("timestamp").drop_duplicates(["user_id", "item_id"], keep="last")
    return log_step("1 clean", d, f"removed {len(df) - len(d)} rows")

def step_kcore(df, k_user=5, k_item=5):
    d = df.copy(); rounds = 0
    while True:
        rounds += 1
        uc = d.user_id.map(d.user_id.value_counts()); ic = d.item_id.map(d.item_id.value_counts())
        keep = (uc >= k_user) & (ic >= k_item)
        if keep.all(): break
        d = d[keep]
    return log_step(f"2 {k_user}/{k_item}-core", d, f"{rounds} rounds")

def step_reindex(df):
    d = df.copy()
    d["uidx"] = d.user_id.astype("category").cat.codes; d["iidx"] = d.item_id.astype("category").cat.codes
    M = csr_matrix((d.rating.values, (d.uidx.values, d.iidx.values)))
    log_step("3 re-index (CSR)", d, f"CSR {M.shape}, nnz={M.nnz}")
    return d, M

def step_split(df, valid=0.1, test=0.1):
    d = df.sort_values(["user_id", "timestamp", "item_id"]).copy()
    d["rk"] = d.groupby("user_id").cumcount(); d["n"] = d.groupby("user_id")["item_id"].transform("size")
    frac = d.rk / d.n
    tr = d[frac < 1 - valid - test]; va = d[(frac >= 1 - valid - test) & (frac < 1 - test)]; te = d[frac >= 1 - test]
    for name, part in [("4 train", tr), ("4 validation", va), ("4 test", te)]: log_step(name, part)
    return tr.drop(columns=["rk", "n"]), va.drop(columns=["rk", "n"]), te.drop(columns=["rk", "n"])

def step_leakage_check(tr, va, te):
    last_tr = tr.groupby("user_id").timestamp.max(); first_va = va.groupby("user_id").timestamp.min()
    first_te = te.groupby("user_id").timestamp.min(); last_va = va.groupby("user_id").timestamp.max()
    bad_va = int((first_va < last_tr.reindex(first_va.index)).sum())
    bad_te = int((first_te < last_va.reindex(first_te.index).fillna(last_tr.reindex(first_te.index))).sum())
    cold_va = len(set(va.item_id) - set(tr.item_id)); cold_te = len(set(te.item_id) - set(tr.item_id) - set(va.item_id))
    return pd.Series({"users with valid before train": bad_va, "users with test before valid": bad_te,
                      "cold-start items in valid": cold_va, "cold-start items in test": cold_te})

K_CORE = 5
log_step("0 raw", ratings)
clean = step_clean(ratings)
core = step_kcore(clean, K_CORE, K_CORE)
core_idx, R_CSR = step_reindex(core)
train_df, valid_df, test_df = step_split(core_idx)
trainvalid_df = pd.concat([train_df, valid_df])
display(pd.DataFrame(DATA_CARD).round(4))
display(step_leakage_check(train_df, valid_df, test_df).to_frame("value"))
# หมายเหตุ: ถ้าผู้ใช้ให้คะแนนหลายเรื่องใน timestamp เดียวกัน (พบได้ใน MovieLens) ลำดับในวินาทีเดียวกันตัดสินด้วย item_id

fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
dc = pd.DataFrame(DATA_CARD)
ax[0].barh(dc.step, dc.ratings, color=["#bbb", "#4C78A8", "#4C78A8", "#4C78A8", "#54A24B", "#F58518", "#E45756"])
ax[0].invert_yaxis(); ax[0].set_title("Data card: ratings after each step")
for name, part, c in [("train", train_df, "#54A24B"), ("validation", valid_df, "#F58518"), ("test", test_df, "#E45756")]:
    ax[1].hist(pd.to_datetime(part.timestamp, unit="s"), bins=50, alpha=.6, color=c, label=name)
ax[1].legend(); ax[1].set_title("Per-user temporal split over calendar time")
plt.tight_layout(); plt.show()

## Part 2 — Core Functions (ชุดเดียวกับ LAB 2)
Model wrapper `SurpriseModel` / `PopularityModel` / `EASEModel` + Metric ทั้ง Accuracy, Ranking และ Beyond-Accuracy
+ ฟังก์ชันใหม่ `evaluate_rating_df` และ `evaluate_lists` (ใช้กับ re-ranking ใน Part 6)

In [ ]:
K = 10                     # Top-K สำหรับ ranking
N_EVAL_USERS = 300         # จำนวนผู้ใช้ที่สุ่มมาประเมิน ranking (ทั้งหมด = None) — ลดเพื่อให้รันเร็วขึ้น
N_NEG = 100                # Sampled Negatives: สุ่มรายการลบ 100 รายการต่อผู้ใช้

def df_to_trainset(df):
    return Dataset.load_from_df(df[["user_id", "item_id", "rating"]], READER).build_full_trainset()

def df_to_testset(df):
    return list(df[["user_id", "item_id", "rating"]].itertuples(index=False, name=None))

class SurpriseModel:
    """ห่ออัลกอริทึมของ Surprise ให้มี scores(inner_u) สำหรับจัดอันดับทุกรายการ"""
    rating_model = True
    def __init__(self, name, factory, family):
        self.name, self.factory, self.family = name, factory, family
    def fit(self, trainset):
        self.ts = trainset; self.algo = self.factory(); self.algo.fit(trainset); return self
    def predict(self, uid, iid):
        return float(self.algo.predict(str(uid), str(iid)).est)
    def scores(self, u):
        a, ts = self.algo, self.ts; mu = ts.global_mean
        if isinstance(a, SVDpp):                                   # vectorized: mu+bu+bi+qi·(pu+|N(u)|^-1/2 Σ yj)
            Iu = [j for j, _ in ts.ur[u]]
            imp = a.yj[Iu].sum(0) / np.sqrt(len(Iu)) if Iu else 0
            return mu + a.bu[u] + a.bi + a.qi @ (a.pu[u] + imp)
        if isinstance(a, SVD):
            return (mu + a.bu[u] + a.bi if a.biased else 0) + a.qi @ a.pu[u]
        if isinstance(a, NMF):
            return (mu + a.bu[u] + a.bi if a.biased else 0) + a.qi @ a.pu[u]
        if isinstance(a, BaselineOnly):
            return mu + a.bu[u] + a.bi
        out = np.empty(ts.n_items)                                 # generic: เรียก estimate ทีละรายการ
        for i in range(ts.n_items):
            try:
                est = a.estimate(u, i)
                out[i] = est[0] if isinstance(est, tuple) else est
            except Exception:                                      # PredictionImpossible → ค่าเฉลี่ยรวม
                out[i] = mu
        return out

class PopularityModel:
    """จำนวนผู้ชอบ (rating>=4) ในข้อมูลฝึก — ranking อย่างเดียว"""
    rating_model, family = False, "Baseline"
    def __init__(self, name="Popularity"): self.name = name
    def fit(self, ts):
        self.ts = ts; self.pop = np.zeros(ts.n_items)
        for _, i, r in ts.all_ratings():
            if r >= LIKE_T: self.pop[i] += 1
        return self
    def predict(self, uid, iid): return np.nan
    def scores(self, u): return self.pop.copy()

class EASEModel:
    """EASE (Steck 2019) บนเมทริกซ์ชอบ 0/1: B = -P/diag(P), diag(B)=0"""
    rating_model, family = False, "Implicit"
    def __init__(self, lam=500.0, name="EASE"): self.lam, self.name = lam, name
    def fit(self, ts):
        self.ts = ts; X = np.zeros((ts.n_users, ts.n_items), dtype=np.float32)
        for u, i, r in ts.all_ratings():
            if r >= LIKE_T: X[u, i] = 1
        P = np.linalg.inv(X.T @ X + self.lam * np.eye(ts.n_items))
        B = -P / np.diag(P); np.fill_diagonal(B, 0); self.X, self.B = X, B
        return self
    def predict(self, uid, iid): return np.nan
    def scores(self, u): return self.X[u] @ self.B

# ---------- Metrics (ชุดเดียวกับ LAB 1 และต้นแบบบทที่ 3) ----------
def precision_at_k(recs, rel, k): return len(set(recs[:k]) & set(rel)) / k if k else 0.0
def recall_at_k(recs, rel, k): rel = set(rel); return len(set(recs[:k]) & rel) / len(rel) if rel else 0.0
def f1_at_k(recs, rel, k):
    p, r = precision_at_k(recs, rel, k), recall_at_k(recs, rel, k); return 0.0 if p + r == 0 else 2*p*r/(p+r)
def hr_at_k(recs, rel, k): return float(bool(set(recs[:k]) & set(rel)))
def reciprocal_rank(recs, rel):
    rel = set(rel)
    for rank, iid in enumerate(recs, 1):
        if iid in rel: return 1.0 / rank
    return 0.0
def ap_at_k(recs, rel, k):
    rel = set(rel); hits = 0; s = 0.0
    for n, iid in enumerate(recs[:k], 1):
        if iid in rel: hits += 1; s += hits / n
    return s / min(len(rel), k) if rel else 0.0
def ndcg_at_k(recs, rel, k):
    rel = set(rel)
    dcg = sum(1/math.log2(n+2) for n, iid in enumerate(recs[:k]) if iid in rel)
    idcg = sum(1/math.log2(n+2) for n in range(min(len(rel), k)))
    return dcg / idcg if idcg else 0.0

def rating_metrics(preds):
    """preds = ผลจาก algo.test(): RMSE, MAE, FCP และสัดส่วนที่ทำนายไม่ได้ (cold start)"""
    impossible = np.mean([p.details.get("was_impossible", False) for p in preds])
    return {"RMSE": accuracy.rmse(preds, verbose=False), "MAE": accuracy.mae(preds, verbose=False),
            "FCP": accuracy.fcp(preds, verbose=False), "Impossible%": 100 * impossible}

# ---------- Beyond-Accuracy ----------
G_INDEX = {g: k for k, g in enumerate(GENRES)}
def genre_vec(iid):
    v = np.zeros(len(GENRES))
    for g in GENRE.get(iid, "unknown").split("|"): v[G_INDEX.get(g, 0)] = 1
    return v
GVEC = {iid: genre_vec(iid) for iid in items.item_id}

def coverage(all_recs, n_catalog): return len(set(all_recs)) / n_catalog
def gini_coefficient(all_recs, catalog):
    counts = pd.Series(all_recs).value_counts()
    x = np.sort(np.array([counts.get(i, 0) for i in catalog], dtype=float)); n = len(x)
    return float((2*np.sum(np.arange(1, n+1)*x)/(n*x.sum())) - (n+1)/n) if x.sum() else 0.0
def ild(rec_ids):
    if len(rec_ids) < 2: return 0.0
    V = np.array([GVEC[i] for i in rec_ids]); V = V / np.maximum(np.linalg.norm(V, axis=1, keepdims=True), 1e-12)
    D = 1 - V @ V.T; return float(D[np.triu_indices(len(rec_ids), 1)].mean())
def novelty(rec_ids, pop, n_users):
    return float(np.mean([-math.log2(max(1, pop.get(i, 1)) / n_users) for i in rec_ids])) if rec_ids else 0.0
def calibration_genre(hist_ids, rec_ids):
    hp = sum(GVEC[i] for i in hist_ids); rp = sum(GVEC[i] for i in rec_ids)
    if np.sum(hp) == 0 or np.sum(rp) == 0: return np.nan
    return float(1 - 0.5*np.abs(hp/hp.sum() - rp/rp.sum()).sum())
def personalization(lists, max_pairs=20000, seed=SEED):
    L = [set(x) for x in lists]; pairs = list(itertools.combinations(range(len(L)), 2))
    if not pairs: return np.nan
    if len(pairs) > max_pairs: pairs = random.Random(seed).sample(pairs, max_pairs)
    return float(1 - np.mean([len(L[a] & L[b]) / max(len(L[a]), 1) for a, b in pairs]))

# ---------- Ranking evaluation engine ----------
def eval_users(test_df, train_ts, n=N_EVAL_USERS, seed=SEED):
    """เลือกผู้ใช้ที่ประเมินได้: อยู่ในชุดฝึก และมีรายการ relevant ในชุดทดสอบ (สุ่ม n คนด้วย seed เดียวกันทุกโมเดล)"""
    rel = test_df[test_df.rating >= LIKE_T]
    cand = sorted(set(rel.user_id) & set(train_ts._raw2inner_id_users), key=int)
    if n and len(cand) > n: cand = sorted(random.Random(seed).sample(cand, n), key=int)
    return cand

def evaluate_ranking(model, train_ts, test_df, users_eval, k=K, n_neg=N_NEG, seed=SEED, keep_lists=False):
    rng = np.random.default_rng(seed)
    inner2raw = np.array([train_ts.to_raw_iid(j) for j in range(train_ts.n_items)])
    pop = {train_ts.to_raw_iid(i): len(v) for i, v in train_ts.ir.items()}
    rel_by_user = test_df[test_df.rating >= LIKE_T].groupby("user_id")["item_id"].apply(set).to_dict()
    rows, all_recs, lists = [], [], {}
    for uid in users_eval:
        u = train_ts.to_inner_uid(uid)
        seen = np.zeros(train_ts.n_items, bool); seen[[j for j, _ in train_ts.ur[u]]] = True
        cand = np.where(~seen)[0]
        rel = {r for r in rel_by_user.get(uid, set()) if r in train_ts._raw2inner_id_items}
        if not rel: continue
        sc_all = model.scores(u); sc = sc_all[cand]
        order = cand[np.argsort(-sc, kind="stable")]
        ranked = list(inner2raw[order]); topk = ranked[:k]
        y = np.isin(inner2raw[cand], list(rel)).astype(int)
        # Sampled protocol: รายการ relevant ทุกตัว + รายการลบที่สุ่ม n_neg ตัว
        neg = cand[y == 0]; neg_s = rng.choice(neg, size=min(n_neg, len(neg)), replace=False)
        pos = cand[y == 1]; sub = np.concatenate([pos, neg_s])
        sub_rank = list(inner2raw[sub[np.argsort(-sc_all[sub], kind="stable")]])
        hist_ids = [train_ts.to_raw_iid(j) for j, _ in train_ts.ur[u]]
        rows.append({"user_id": uid, "n_rel": len(rel),
            f"Precision@{k}": precision_at_k(topk, rel, k), f"Recall@{k}": recall_at_k(topk, rel, k),
            f"F1@{k}": f1_at_k(topk, rel, k), f"HR@{k}": hr_at_k(topk, rel, k),
            "MRR": reciprocal_rank(ranked, rel), f"MAP@{k}": ap_at_k(topk, rel, k), f"nDCG@{k}": ndcg_at_k(topk, rel, k),
            "AUC": roc_auc_score(y, sc) if 0 < y.sum() < len(y) else np.nan,
            f"HR@{k}(sampled)": hr_at_k(sub_rank[:k], rel, k), f"nDCG@{k}(sampled)": ndcg_at_k(sub_rank[:k], rel, k),
            "ILD": ild(topk), "Novelty": novelty(topk, pop, train_ts.n_users),
            "Calibration": calibration_genre(hist_ids, topk), "ARP": float(np.mean([pop.get(i, 0) for i in topk]))})
        all_recs += topk; lists[uid] = topk
    detail = pd.DataFrame(rows)
    summary = detail.drop(columns=["user_id"]).mean(numeric_only=True).to_dict()
    catalog = list(inner2raw)
    summary.update({"Coverage": coverage(all_recs, len(catalog)), "Gini": gini_coefficient(all_recs, catalog),
                    "Personalization": personalization(list(lists.values())), "n_users": len(detail)})
    return (pd.Series(summary), detail, lists) if keep_lists else (pd.Series(summary), detail)

def evaluate_rating_df(model, df):
    preds = model.algo.test(df_to_testset(df)); return rating_metrics(preds)

def evaluate_lists(lists, ts_, test_df_, k=K):
    """ประเมินรายการแนะนำที่สร้างเสร็จแล้ว (ใช้กับ pipeline ที่มีการ re-rank)"""
    rel_by_user = test_df_[test_df_.rating >= LIKE_T].groupby("user_id")["item_id"].apply(set).to_dict()
    pop = {ts_.to_raw_iid(i): len(v) for i, v in ts_.ir.items()}; rows = []; all_recs = []
    for uid, top in lists.items():
        rel = rel_by_user.get(uid, set())
        if not rel: continue
        rows.append({f"Precision@{k}": precision_at_k(top, rel, k), f"Recall@{k}": recall_at_k(top, rel, k),
                     f"nDCG@{k}": ndcg_at_k(top, rel, k), f"HR@{k}": hr_at_k(top, rel, k), "ILD": ild(top),
                     "Novelty": novelty(top, pop, ts_.n_users), "ARP": np.mean([pop.get(i, 0) for i in top])})
        all_recs += top
    s = pd.DataFrame(rows).mean()
    s["Coverage"] = coverage(all_recs, ts_.n_items); s["Personalization"] = personalization(list(lists.values()))
    return s

TS_TRAIN = df_to_trainset(train_df)
TS_TRAINVALID = df_to_trainset(trainvalid_df)
USERS_VALID = eval_users(valid_df, TS_TRAIN, n=N_EVAL_USERS)
USERS_TEST = eval_users(test_df, TS_TRAINVALID, n=N_EVAL_USERS)
print(f"Train={TS_TRAIN.n_ratings}  Valid={len(valid_df)}  Test={len(test_df)} | eval users valid={len(USERS_VALID)} test={len(USERS_TEST)}")

## Part 3 — ระบบฐานก่อนปรับจูน (Default hyperparameters) บน Validation
ทุกการตัดสินใจเรื่องไฮเพอร์พารามิเตอร์ใช้ **Validation เท่านั้น** — Test ถูกเก็บไว้ใช้ครั้งเดียวใน Part 5

In [ ]:
DEFAULT_FACTORIES = {
    "Popularity":       (lambda: PopularityModel(), "Baseline"),
    "BaselineOnly":     (lambda: SurpriseModel("BaselineOnly", lambda: BaselineOnly(verbose=False), "Baseline"), "Baseline"),
    "UserKNN-Means":    (lambda: SurpriseModel("UserKNN-Means", lambda: KNNWithMeans(verbose=False), "Memory-based"), "Memory-based"),
    "ItemKNN-Baseline": (lambda: SurpriseModel("ItemKNN-Baseline", lambda: KNNBaseline(sim_options={"name": "pearson_baseline", "user_based": False}, verbose=False), "Memory-based"), "Memory-based"),
    "SVD":              (lambda: SurpriseModel("SVD", lambda: SVD(random_state=SEED), "Latent factor"), "Latent factor"),
    "SVD++":            (lambda: SurpriseModel("SVD++", lambda: SVDpp(n_factors=20, random_state=SEED), "Latent factor"), "Latent factor"),
    "NMF":              (lambda: SurpriseModel("NMF", lambda: NMF(random_state=SEED), "Latent factor"), "Latent factor"),
    "EASE":             (lambda: EASEModel(lam=500.0), "Implicit"),
}
default_rows, DEFAULT_FIT = [], {}
for name, (fac, fam) in DEFAULT_FACTORIES.items():
    t0 = time.time(); m = fac(); m.name = name; m.fit(TS_TRAIN); DEFAULT_FIT[name] = m
    rmet = evaluate_rating_df(m, valid_df) if m.rating_model else {}
    s, _ = evaluate_ranking(m, TS_TRAIN, valid_df, USERS_VALID)
    default_rows.append({"Model": name, "Family": fam, "RMSE": rmet.get("RMSE", np.nan), "MAE": rmet.get("MAE", np.nan),
                         f"HR@{K}": s[f"HR@{K}"], f"nDCG@{K}": s[f"nDCG@{K}"], "MRR": s["MRR"], "Coverage": s["Coverage"],
                         "sec": time.time() - t0})
default_valid = pd.DataFrame(default_rows)
display(default_valid.sort_values(f"nDCG@{K}", ascending=False).round(4).reset_index(drop=True))

## Part 4 — การปรับ Hyperparameter
| วิธี | เครื่องมือ | Objective | ข้อมูลที่ใช้ |
|---|---|---|---|
| 4.1 Grid search | `surprise.model_selection.GridSearchCV` | RMSE/MAE (3-fold CV) | Train |
| 4.2 Random search | `RandomizedSearchCV` | RMSE (3-fold CV) | Train |
| 4.3 Hold-out tuning | loop เขียนเอง | **RMSE และ nDCG@10** | Train → Validation |
| 4.4 Overfitting curve | จำนวนรอบของ SVD | RMSE train vs valid | Train → Validation |

In [ ]:
# 4.1 GridSearchCV — SVD (หัวข้อ 5.9 + ตารางที่ 5.20)
data_train = Dataset.load_from_df(train_df[["user_id", "item_id", "rating"]], READER)
param_grid = {"n_factors": [50, 100], "n_epochs": [20, 40], "lr_all": [0.005, 0.01], "reg_all": [0.02, 0.1]}
t0 = time.time()
gs = GridSearchCV(SVD, param_grid, measures=["rmse", "mae"], cv=3, n_jobs=1, joblib_verbose=0)
gs.fit(data_train)
print(f"GridSearchCV: {len(gs.cv_results['params'])} configs × 3 folds in {time.time()-t0:.1f}s")
print("best RMSE:", round(gs.best_score["rmse"], 4), gs.best_params["rmse"])
cvres = pd.DataFrame(gs.cv_results)
grid_table = pd.concat([pd.json_normalize(cvres["params"]), cvres[["mean_test_rmse", "std_test_rmse", "mean_test_mae", "mean_fit_time"]]], axis=1)
display(grid_table.sort_values("mean_test_rmse").head(8).round(4))
BEST_SVD_RMSE = gs.best_params["rmse"]

In [ ]:
# 4.2 RandomizedSearchCV — KNNBaseline (หัวข้อ 5.6: k, ความคล้าย, การหด)
param_dist = {"k": [20, 40, 60, 80], "min_k": [1, 3, 5], "verbose": [False],
              "sim_options": {"name": ["pearson_baseline", "msd", "cosine"], "user_based": [False], "shrinkage": [50, 100]}}
t0 = time.time()
rs = RandomizedSearchCV(KNNBaseline, param_dist, n_iter=8, measures=["rmse"], cv=3, random_state=SEED, n_jobs=1)
rs.fit(data_train)
print(f"RandomizedSearchCV: 8 configs × 3 folds in {time.time()-t0:.1f}s")
print("best RMSE:", round(rs.best_score["rmse"], 4), rs.best_params["rmse"])
rsres = pd.DataFrame(rs.cv_results)
display(pd.concat([pd.json_normalize(rsres["params"]), rsres[["mean_test_rmse", "std_test_rmse", "mean_fit_time"]]], axis=1)
        .sort_values("mean_test_rmse").round(4))
BEST_KNN_RMSE = rs.best_params["rmse"]

In [ ]:
# 4.3 Hold-out tuning บน Validation: วัดทั้ง RMSE และ nDCG@10 → objective ต่างกัน เลือก config ต่างกันได้
N_TUNE_USERS = 200
USERS_TUNE = USERS_VALID[:N_TUNE_USERS]
tune_rows = []
def tune(model_name, factory, params):
    t0 = time.time(); m = factory(); m.fit(TS_TRAIN)
    r = evaluate_rating_df(m, valid_df)["RMSE"] if m.rating_model else np.nan
    s, _ = evaluate_ranking(m, TS_TRAIN, valid_df, USERS_TUNE)
    tune_rows.append({"Model": model_name, **params, "RMSE": r, f"nDCG@{K}": s[f"nDCG@{K}"], f"HR@{K}": s[f"HR@{K}"],
                      "Coverage": s["Coverage"], "sec": time.time() - t0})

for biased in [True, False]:                     # biased=False: ตัด b_u, b_i ออก → ไม่ดันรายการคะแนนเฉลี่ยสูงแต่มีคนดูน้อย
    for nf in [50, 100, 200]:
        for reg in [0.02, 0.1]:
            tune("SVD", lambda: SurpriseModel("SVD", lambda: SVD(n_factors=nf, reg_all=reg, biased=biased, n_epochs=20, random_state=SEED), "Latent factor"),
                 {"biased": biased, "n_factors": nf, "reg_all": reg})
for k_ in [10, 40]:
    for ms in [1, 10]:
        tune("ItemKNN-Baseline", lambda: SurpriseModel("ItemKNN-Baseline", lambda: KNNBaseline(
            k=k_, sim_options={"name": "pearson_baseline", "user_based": False, "min_support": ms, "shrinkage": 100}, verbose=False), "Memory-based"),
             {"k": k_, "min_support": ms})
for lam in [50, 200, 500, 1000, 3000]:
    tune("EASE", lambda: EASEModel(lam=lam), {"lambda": lam})
tune_results = pd.DataFrame(tune_rows)
display(tune_results.round(4))

best_by = {}
for name, g in tune_results.groupby("Model"):
    pcols = [c for c in g.columns if c not in ("Model", "RMSE", f"nDCG@{K}", f"HR@{K}", "Coverage", "sec") and g[c].notna().any()]
    best_r = g.loc[g.RMSE.idxmin(), pcols].to_dict() if g.RMSE.notna().any() else None
    best_n = g.loc[g[f"nDCG@{K}"].idxmax(), pcols].to_dict()
    best_by[name] = {"best for RMSE": best_r, f"best for nDCG@{K}": best_n}
display(pd.DataFrame(best_by).T)
# สังเกต: config ที่ดีที่สุดสำหรับ RMSE กับ nDCG ไม่จำเป็นต้องเป็นตัวเดียวกัน → เลือก objective ให้ตรงกับงานจริง (5.17.2)
# เช่น SVD แบบ biased=False ได้ RMSE แย่ลงแต่ nDCG ดีขึ้นชัดเจน เพราะ b_i ไม่ดันรายการที่มีคะแนนเฉลี่ยสูงจากคนดูไม่กี่คนขึ้นมาบนสุด
# ส่วน min_support ของ KNN ไม่ช่วยในข้อมูลชุดนี้ — การปรับจูนต้องวัดจริง ไม่ใช่เดา

In [ ]:
# 4.4 Overfitting curve: จำนวนรอบของ SVD (reg ต่ำ) — RMSE บน Train ลดเรื่อย ๆ แต่บน Validation กลับเพิ่ม (5.9.3)
curve = []
for ep in [1, 5, 10, 20, 40, 60, 80]:
    a = SVD(n_factors=100, n_epochs=ep, lr_all=0.01, reg_all=0.005, random_state=SEED).fit(TS_TRAIN)
    tr_pred = a.test(TS_TRAIN.build_testset()); va_pred = a.test(df_to_testset(valid_df))
    curve.append({"epochs": ep, "train RMSE": accuracy.rmse(tr_pred, verbose=False), "valid RMSE": accuracy.rmse(va_pred, verbose=False)})
curve = pd.DataFrame(curve); display(curve.round(4))

fig, ax = plt.subplots(1, 4, figsize=(21, 4))
ax[0].plot(curve.epochs, curve["train RMSE"], marker="o", label="train"); ax[0].plot(curve.epochs, curve["valid RMSE"], marker="o", label="validation")
ax[0].set_xlabel("epochs"); ax[0].legend(); ax[0].set_title("4.4 SVD overfitting curve")
hm = grid_table.pivot_table(index="reg_all", columns="lr_all", values="mean_test_rmse", aggfunc="min")
im = ax[1].imshow(hm.values, cmap="viridis_r"); ax[1].set_xticks(range(hm.shape[1])); ax[1].set_xticklabels(hm.columns)
ax[1].set_yticks(range(hm.shape[0])); ax[1].set_yticklabels(hm.index); ax[1].set_xlabel("lr_all"); ax[1].set_ylabel("reg_all")
for a_ in range(hm.shape[0]):
    for b_ in range(hm.shape[1]): ax[1].text(b_, a_, f"{hm.values[a_, b_]:.4f}", ha="center", va="center", color="w")
ax[1].set_title("4.1 GridSearchCV best RMSE")
sv = tune_results[(tune_results.Model == "SVD") & (tune_results.reg_all == 0.02)]
for b, g in sv.groupby("biased"):
    ax[2].plot(g.n_factors, g.RMSE, marker="o", label=f"RMSE biased={b}")
ax2b = ax[2].twinx()
for b, g in sv.groupby("biased"):
    ax2b.plot(g.n_factors, g[f"nDCG@{K}"], marker="s", ls="--", label=f"nDCG biased={b}")
ax[2].set_xlabel("n_factors"); ax[2].set_ylabel("valid RMSE"); ax2b.set_ylabel(f"valid nDCG@{K}")
ax[2].legend(loc="upper left", fontsize=8); ax2b.legend(loc="lower right", fontsize=8); ax[2].set_title("4.3 SVD (reg=0.02): RMSE vs nDCG objective")
ea = tune_results[tune_results.Model == "EASE"]
ax[3].plot(ea["lambda"], ea[f"nDCG@{K}"], marker="o"); ax[3].set_xscale("log"); ax[3].set_xlabel("λ"); ax[3].set_title("4.3 EASE λ (validation nDCG)")
plt.tight_layout(); plt.show()

## Part 5 — Refit บน Train+Validation แล้วประเมิน **ครั้งเดียว** บน Test
เทียบ Default กับ Tuned ของแต่ละตระกูล และ SVD ที่ปรับเพื่อ RMSE กับ SVD ที่ปรับเพื่อ nDCG

In [ ]:
def p_int(d, k): return int(d[k]) if d and k in d and pd.notna(d[k]) else None
svd_ndcg = best_by["SVD"][f"best for nDCG@{K}"]; knn_ndcg = best_by["ItemKNN-Baseline"][f"best for nDCG@{K}"]
ease_lam = float(best_by["EASE"][f"best for nDCG@{K}"]["lambda"])
knn_rs = dict(BEST_KNN_RMSE); knn_rs_sim = knn_rs.pop("sim_options"); knn_rs.pop("verbose", None)

FINAL_FACTORIES = {
    "Popularity": lambda: PopularityModel(),
    "BaselineOnly": DEFAULT_FACTORIES["BaselineOnly"][0],
    "ItemKNN-Baseline (default)": DEFAULT_FACTORIES["ItemKNN-Baseline"][0],
    "ItemKNN-Baseline (tuned RMSE)": lambda: SurpriseModel("ItemKNN-Baseline (tuned RMSE)", lambda: KNNBaseline(**knn_rs, sim_options=knn_rs_sim, verbose=False), "Memory-based"),
    "ItemKNN-Baseline (tuned nDCG)": lambda: SurpriseModel("ItemKNN-Baseline (tuned nDCG)", lambda: KNNBaseline(
        k=p_int(knn_ndcg, "k"), sim_options={"name": "pearson_baseline", "user_based": False, "min_support": p_int(knn_ndcg, "min_support"), "shrinkage": 100}, verbose=False), "Memory-based"),
    "SVD (default)": DEFAULT_FACTORIES["SVD"][0],
    "SVD (tuned RMSE)": lambda: SurpriseModel("SVD (tuned RMSE)", lambda: SVD(**BEST_SVD_RMSE, random_state=SEED), "Latent factor"),
    "SVD (tuned nDCG)": lambda: SurpriseModel("SVD (tuned nDCG)", lambda: SVD(n_factors=p_int(svd_ndcg, "n_factors"), reg_all=float(svd_ndcg["reg_all"]),
                                                                       biased=bool(svd_ndcg["biased"]), random_state=SEED), "Latent factor"),
    "SVD++": DEFAULT_FACTORIES["SVD++"][0],
    "EASE (default λ=500)": DEFAULT_FACTORIES["EASE"][0],
    f"EASE (tuned λ={ease_lam:g})": lambda: EASEModel(lam=ease_lam),
}
FINAL_NAMES = list(FINAL_FACTORIES)
FINAL, final_rows, FINAL_DETAIL, FINAL_LISTS = {}, [], {}, {}
for name, fac in FINAL_FACTORIES.items():
    t0 = time.time(); m = fac(); m.name = name; m.fit(TS_TRAINVALID); FINAL[name] = m
    rmet = evaluate_rating_df(m, test_df) if m.rating_model else {}
    s, d, lists = evaluate_ranking(m, TS_TRAINVALID, test_df, USERS_TEST, keep_lists=True)
    FINAL_DETAIL[name], FINAL_LISTS[name] = d, lists
    final_rows.append({"Model": name, "Family": getattr(m, "family", ""), "RMSE": rmet.get("RMSE", np.nan), "MAE": rmet.get("MAE", np.nan),
                       "FCP": rmet.get("FCP", np.nan), **{c: s[c] for c in s.index if c != "n_users"}, "sec": time.time() - t0})
    print(f"{name:32s} done in {time.time()-t0:.1f}s")
final_results = pd.DataFrame(final_rows)
ACC_COLS = ["RMSE", "MAE", f"Precision@{K}", f"Recall@{K}", f"F1@{K}", f"HR@{K}", "MRR", f"MAP@{K}", f"nDCG@{K}", "AUC"]
BEYOND_COLS = ["Coverage", "Gini", "ILD", "Novelty", "Calibration", "Personalization", "ARP"]
print("Test — Accuracy (Rating + Ranking)")
display(final_results[["Model"] + ACC_COLS].round(4))
print("Test — Beyond-Accuracy")
display(final_results[["Model"] + BEYOND_COLS].round(4))
# สังเกต: (1) SVD ที่ปรับเพื่อ RMSE ได้ RMSE ดีที่สุด แต่ nDCG ต่ำ ส่วน SVD ที่ปรับเพื่อ nDCG กลับกัน → objective ของการปรับจูนกำหนดผล
# (2) EASE ยังเป็นฐานที่แข็งแรงที่สุดสำหรับ ranking (5.12.2) และ λ ที่ปรับแล้วเพิ่มทั้ง nDCG และ Coverage
# (3) config ที่ชนะบน Validation อาจไม่ชนะบน Test เสมอไป (เช่น KNN) — ความต่างเล็ก ๆ ควรตรวจด้วย bootstrap/หลาย seed

## Part 6 — Prediction Pipeline: แนะนำ + Re-ranking ความหลากหลาย + Cold-start + Save/Load
รวมทุกขั้นเป็น object เดียวที่พร้อมใช้งาน:
`fit(df)` → `predict_rating(user, movie)` → `recommend(user, k, diversity)`
- **กรองเรื่องที่ดูแล้ว** · **ผู้ใช้ใหม่ → Popularity (cold start)**
- **MMR re-ranking**: เลือกทีละเรื่องโดยชั่ง `(1−λ)·คะแนน − λ·ความคล้ายแนวกับเรื่องที่เลือกแล้ว`

In [ ]:
class RecPipeline:
    def __init__(self, ranker_factory, rater_factory=None, pool=100):
        self.ranker_factory, self.rater_factory, self.pool = ranker_factory, rater_factory, pool
    def fit(self, df):
        self.ts = df_to_trainset(df)
        self.ranker = self.ranker_factory().fit(self.ts)
        self.rater = self.rater_factory().fit(self.ts) if self.rater_factory else None
        self.pop = PopularityModel().fit(self.ts); return self
    def predict_rating(self, uid, iid):
        return self.rater.predict(uid, iid) if self.rater else np.nan
    def _candidates(self, uid):
        if uid in self.ts._raw2inner_id_users:
            u = self.ts.to_inner_uid(uid); sc = self.ranker.scores(u); seen = {j for j, _ in self.ts.ur[u]}
        else:                                                          # cold-start user → popularity
            sc = self.pop.scores(0); seen = set()
        order = [j for j in np.argsort(-sc, kind="stable") if j not in seen][:self.pool]
        return [self.ts.to_raw_iid(j) for j in order], np.array([sc[j] for j in order])
    def recommend(self, uid, k=10, diversity=0.0):
        ids, sc = self._candidates(uid)
        s = (sc - sc.min()) / (sc.max() - sc.min() + 1e-12)
        V = np.array([GVEC[i] for i in ids]); V = V / np.maximum(np.linalg.norm(V, axis=1, keepdims=True), 1e-12)
        chosen, remain = [], list(range(len(ids)))
        while remain and len(chosen) < k:
            if chosen and diversity > 0:
                sim = (V[remain] @ V[chosen].T).max(axis=1)
                val = (1 - diversity) * s[remain] - diversity * sim
            else:
                val = s[remain]
            b = remain[int(np.argmax(val))]; chosen.append(b); remain.remove(b)
        return [ids[j] for j in chosen]

# object ที่เก็บ lambda (factory) pickle ไม่ได้ → ตัด factory ออกตอนบันทึก (โมเดลที่ฝึกแล้วยังอยู่ครบ)
SurpriseModel.__getstate__ = lambda self: {k: v for k, v in self.__dict__.items() if k != "factory"}
RecPipeline.__getstate__ = lambda self: {k: v for k, v in self.__dict__.items() if not k.endswith("_factory")}

best_rank_name = final_results.sort_values(f"nDCG@{K}", ascending=False).Model.iloc[0]
best_rate_name = final_results.dropna(subset=["RMSE"]).sort_values("RMSE").Model.iloc[0]
print("Ranker =", best_rank_name, "| Rater =", best_rate_name)
PIPE = RecPipeline(FINAL_FACTORIES[best_rank_name], FINAL_FACTORIES[best_rate_name]).fit(trainvalid_df)

# 6.1 Diversity sweep: nDCG vs ILD vs Coverage บน Test
div_rows = []
for lam in [0.0, 0.1, 0.2, 0.3, 0.5, 0.7]:
    lists = {uid: PIPE.recommend(uid, K, lam) for uid in USERS_TEST}
    div_rows.append({"λ diversity": lam, **evaluate_lists(lists, PIPE.ts, test_df).to_dict()})
div_results = pd.DataFrame(div_rows); display(div_results.round(4))

# 6.2 Save / Load (พร้อมนำไปใช้งาน)
with open("rec_pipeline.pkl", "wb") as f: pickle.dump(PIPE, f)
with open("rec_pipeline.pkl", "rb") as f: PIPE2 = pickle.load(f)
uid0 = USERS_TEST[0]
print("Saved & reloaded. Same output:", PIPE.recommend(uid0, 5) == PIPE2.recommend(uid0, 5))
print("Cold-start user 'new_user' →", [TITLE[i] for i in PIPE.recommend("new_user", 5)])

## Part 7 — ตัวอย่าง Output บน Validation และ Test: เทียบ Rating และ Ranking ทุกโมเดล
- 7.1 **Validation** (โมเดลก่อนปรับจูน ฝึกบน Train): ค่าทำนายเทียบค่าจริง
- 7.2 **Test** (โมเดลหลังปรับจูน ฝึกบน Train+Validation): ค่าทำนายเทียบค่าจริง
- 7.3 **Test Ranking**: Top-10 ของแต่ละโมเดล (✓ = ผู้ใช้ชอบจริงใน Test)

In [ ]:
def short(iid, n=28):
    t = TITLE.get(iid, iid); return t if len(t) <= n else t[:n-1] + "…"
rel_cnt = test_df[test_df.rating >= LIKE_T].groupby("user_id").size()
SAMPLE_USERS = [u for u in rel_cnt.sort_values(ascending=False).index if u in USERS_TEST and u in set(valid_df.user_id)][:3]
UINFO = users.set_index("user_id")

uid = SAMPLE_USERS[0]
print(f"7.1 Validation — ผู้ใช้ {uid} ({UINFO.loc[uid].age} ปี, {UINFO.loc[uid].gender}, {UINFO.loc[uid].occupation})")
sub = valid_df[valid_df.user_id == uid].head(8)
tab = pd.DataFrame({"movie": [short(i) for i in sub.item_id], "true": sub.rating.values})
for name, m in DEFAULT_FIT.items():
    if m.rating_model: tab[name] = [m.predict(uid, i) for i in sub.item_id]
display(tab.round(2))

print(f"7.2 Test — ผู้ใช้ {uid}")
sub = test_df[test_df.user_id == uid].head(8)
tab = pd.DataFrame({"movie": [short(i) for i in sub.item_id], "true": sub.rating.values})
for name, m in FINAL.items():
    if m.rating_model: tab[name] = [m.predict(uid, i) for i in sub.item_id]
tab["Pipeline"] = [PIPE.predict_rating(uid, i) for i in sub.item_id]
display(tab.round(2))

for uid in SAMPLE_USERS:
    rel = set(test_df[(test_df.user_id == uid) & (test_df.rating >= LIKE_T)].item_id)
    comp = pd.DataFrame({name: [("✓ " if i in rel else "") + short(i) for i in FINAL_LISTS[name][uid]] for name in FINAL_NAMES},
                        index=[f"#{n}" for n in range(1, K+1)]).T
    comp.loc["Pipeline (λ=0.3)"] = [("✓ " if i in rel else "") + short(i) for i in PIPE.recommend(uid, K, 0.3)]
    comp["hits"] = [sum(x.startswith("✓") for x in row) for row in comp.values]
    print(f"\n7.3 Test ranking — ผู้ใช้ {uid}: relevant {len(rel)} เรื่อง"); display(comp)

## Part 8 — ตารางสรุปและ Chart

In [ ]:
summary = final_results[["Model", "RMSE", "MAE", f"HR@{K}", "MRR", f"nDCG@{K}", "AUC"] + BEYOND_COLS]
display(summary.sort_values(f"nDCG@{K}", ascending=False).round(4).reset_index(drop=True))
gain = []
for fam, d_name, t_name in [("ItemKNN-Baseline", "ItemKNN-Baseline (default)", "ItemKNN-Baseline (tuned nDCG)"),
                            ("SVD", "SVD (default)", "SVD (tuned nDCG)"), ("EASE", "EASE (default λ=500)", f"EASE (tuned λ={ease_lam:g})")]:
    a, b = final_results.set_index("Model").loc[d_name], final_results.set_index("Model").loc[t_name]
    gain.append({"Family": fam, f"nDCG default": a[f"nDCG@{K}"], f"nDCG tuned": b[f"nDCG@{K}"],
                 "RMSE default": a["RMSE"], "RMSE tuned": b["RMSE"], "Coverage default": a["Coverage"], "Coverage tuned": b["Coverage"]})
display(pd.DataFrame(gain).round(4))

fig, axes = plt.subplots(2, 3, figsize=(21, 10))
fr = final_results.set_index("Model")
fr["RMSE"].dropna().sort_values().plot.barh(ax=axes[0, 0], color="#4C78A8"); axes[0, 0].invert_yaxis()
axes[0, 0].set_xlim(0.85, fr["RMSE"].max() + .03); axes[0, 0].set_title("Test RMSE (lower is better)")
fr[[f"HR@{K}", "MRR", f"nDCG@{K}"]].sort_values(f"nDCG@{K}").plot.barh(ax=axes[0, 1]); axes[0, 1].set_title(f"Test ranking @{K}")
g = pd.DataFrame(gain).set_index("Family")
g[["nDCG default", "nDCG tuned"]].plot.bar(ax=axes[0, 2], rot=0, color=["#bbb", "#54A24B"]); axes[0, 2].set_title("Default vs tuned (nDCG)")
bm = fr[["Coverage", "Gini", "ILD", "Calibration", "Personalization"]]
axes[1, 0].imshow(bm.values, cmap="viridis", aspect="auto", vmin=0, vmax=1)
axes[1, 0].set_xticks(range(bm.shape[1])); axes[1, 0].set_xticklabels(bm.columns, rotation=20)
axes[1, 0].set_yticks(range(bm.shape[0])); axes[1, 0].set_yticklabels(bm.index, fontsize=8)
for a_ in range(bm.shape[0]):
    for b_ in range(bm.shape[1]): axes[1, 0].text(b_, a_, f"{bm.values[a_, b_]:.2f}", ha="center", va="center", color="w", fontsize=8)
axes[1, 0].set_title("Test beyond-accuracy")
for name, r in fr.iterrows():
    axes[1, 1].scatter(r["Coverage"], r[f"nDCG@{K}"], s=20 + 10*r["Novelty"])
    axes[1, 1].annotate(name, (r["Coverage"], r[f"nDCG@{K}"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
axes[1, 1].set_xlabel("Coverage"); axes[1, 1].set_ylabel(f"nDCG@{K}"); axes[1, 1].set_title("Accuracy vs coverage")
axes[1, 2].plot(div_results["λ diversity"], div_results[f"nDCG@{K}"], marker="o", label=f"nDCG@{K}")
axes[1, 2].plot(div_results["λ diversity"], div_results["ILD"], marker="s", label="ILD")
axes[1, 2].plot(div_results["λ diversity"], div_results["Coverage"], marker="^", label="Coverage")
axes[1, 2].set_xlabel("MMR λ"); axes[1, 2].legend(); axes[1, 2].set_title("Pipeline re-ranking trade-off")
plt.tight_layout(); plt.show()

## Part 9 — GUI: ใช้งาน Prediction Pipeline
- **ผู้ใช้เดิม**: ข้อมูลผู้ใช้ ประวัติ และรายการแนะนำจากโมเดลที่ปรับจูนแล้ว พร้อมปรับความหลากหลาย (MMR λ)
- **เทียบโมเดล**: Top-K ของหลายโมเดลสำหรับผู้ใช้คนเดียวกัน (✓ = ชอบจริงใน Test)
- **อธิบายค่าทำนาย**: แยกค่าทำนายของ SVD เป็น μ + b_u + b_i + q_i·p_u (หัวข้อ 5.9.4)
- **ผู้ใช้ใหม่ (Cold start)**: เลือกเรื่องที่ชอบ → EASE ให้คะแนนทันทีจาก x·B โดยไม่ต้องฝึกใหม่

In [ ]:
import gradio as gr
USER_CHOICES = sorted(trainvalid_df.user_id.unique(), key=int)
MOVIE_CHOICES = [f"{i} | {TITLE[i]}" for i in sorted(core.item_id.unique(), key=int)]
SVD_FOR_EXPLAIN = FINAL["SVD (tuned RMSE)"]
EASE_FINAL = FINAL[f"EASE (tuned λ={ease_lam:g})"]

def gui_user(uid, model_name, k, lam):
    info = UINFO.loc[uid]; ts_ = TS_TRAINVALID; u = ts_.to_inner_uid(uid)
    hist = [(ts_.to_raw_iid(j), r) for j, r in ts_.ur[u]]
    liked = [i for i, r in hist if r >= LIKE_T]
    gtop = pd.Series([x for i in liked for x in GENRE[i].split("|")]).value_counts().head(5)
    fut = test_df[test_df.user_id == uid]; rel = set(fut[fut.rating >= LIKE_T].item_id)
    md = (f"### ผู้ใช้ {uid}\n- อายุ **{info.age}** · เพศ **{info.gender}** · อาชีพ **{info.occupation}**\n"
          f"- ประวัติ (Train+Validation): **{len(hist)}** คะแนน · เฉลี่ย **{np.mean([r for _, r in hist]):.2f}** · ชอบ **{len(liked)}** เรื่อง\n"
          f"- แนวที่ชอบ: {', '.join(f'{a} ({b})' for a, b in gtop.items())}\n- Test (อนาคต): {len(fut)} คะแนน, ชอบ {len(rel)} เรื่อง")
    h = pd.DataFrame([{"Movie": TITLE[i], "Genre": GENRE[i], "Rating": r} for i, r in hist]).sort_values("Rating", ascending=False).head(15)
    pipe = PIPE if model_name == "Pipeline (best)" else RecPipeline(lambda: FINAL[model_name], None)
    if pipe is not PIPE:
        pipe.ts, pipe.ranker, pipe.pop = TS_TRAINVALID, FINAL[model_name], PIPE.pop
    recs = pipe.recommend(uid, int(k), float(lam))
    pop = {ts_.to_raw_iid(i): len(v) for i, v in ts_.ir.items()}
    rec = pd.DataFrame([{"Rank": n, "Movie": TITLE[i], "Year": YEAR.get(i), "Genre": GENRE[i],
                         "Pred. rating": round(PIPE.predict_rating(uid, i), 2), "#ratings": pop.get(i, 0),
                         "Liked in test": "✓" if i in rel else ""} for n, i in enumerate(recs, 1)])
    return md, h, rec

def gui_compare(uid, k):
    fut = test_df[test_df.user_id == uid]; rel = set(fut[fut.rating >= LIKE_T].item_id)
    ts_ = TS_TRAINVALID; u = ts_.to_inner_uid(uid); seen = {j for j, _ in ts_.ur[u]}
    out = {}
    for name, m in FINAL.items():
        sc = m.scores(u); top = [ts_.to_raw_iid(j) for j in np.argsort(-sc, kind="stable") if j not in seen][:int(k)]
        out[name] = [("✓ " if i in rel else "") + TITLE[i] for i in top]
    return pd.DataFrame(out, index=[f"#{n}" for n in range(1, int(k)+1)]).T.reset_index(names="Model")

def gui_explain(uid, movie):
    iid = movie.split(" | ")[0]; a = SVD_FOR_EXPLAIN.algo; ts_ = TS_TRAINVALID
    true = core[(core.user_id == uid) & (core.item_id == iid)].rating
    if uid not in ts_._raw2inner_id_users or iid not in ts_._raw2inner_id_items:
        return "ผู้ใช้หรือภาพยนตร์ไม่อยู่ในข้อมูลฝึก (cold start)", pd.DataFrame()
    u, i = ts_.to_inner_uid(uid), ts_.to_inner_iid(iid)
    parts = {"μ (global mean)": ts_.global_mean, "b_u (user bias)": a.bu[u], "b_i (item bias)": a.bi[i], "q_i·p_u (interaction)": a.qi[i] @ a.pu[u]}
    total = sum(parts.values())
    md = (f"**{TITLE[iid]}** ({GENRE[iid]}) — SVD ทำนาย **{np.clip(total, 1, 5):.3f}** "
          f"(จริง: {true.iloc[0] if len(true) else 'ยังไม่เคยให้คะแนน'})")
    df = pd.DataFrame({"component": list(parts) + ["sum (clipped 1–5)"], "value": list(parts.values()) + [np.clip(total, 1, 5)]}).round(4)
    preds = pd.DataFrame([{"component": f"{n}", "value": round(m.predict(uid, iid), 3)} for n, m in FINAL.items() if m.rating_model])
    return md, pd.concat([df, preds])

def gui_newuser(liked_movies, k):
    ts_ = TS_TRAINVALID; x = np.zeros(ts_.n_items, dtype=np.float32); chosen = []
    for mv in liked_movies or []:
        iid = mv.split(" | ")[0]
        if iid in ts_._raw2inner_id_items: x[ts_.to_inner_iid(iid)] = 1; chosen.append(iid)
    if not chosen:
        top = [ts_.to_raw_iid(j) for j in np.argsort(-PIPE.pop.pop)[:int(k)]]; src = "Popularity (ยังไม่ได้เลือกเรื่องที่ชอบ)"
    else:
        sc = x @ EASE_FINAL.B; sc[x > 0] = -np.inf
        top = [ts_.to_raw_iid(j) for j in np.argsort(-sc)[:int(k)]]; src = f"EASE จาก {len(chosen)} เรื่องที่ชอบ"
    return src, pd.DataFrame([{"Rank": n, "Movie": TITLE[i], "Genre": GENRE[i]} for n, i in enumerate(top, 1)])

with gr.Blocks(title="Chapter 5 — CF Pipeline (MovieLens-100K)") as demo:
    gr.Markdown("## Chapter 5 — End-to-End Collaborative Filtering Pipeline (MovieLens-100K, tuned models)")
    with gr.Tab("ผู้ใช้เดิม"):
        with gr.Row():
            u_dd = gr.Dropdown(USER_CHOICES, value=SAMPLE_USERS[0], label="User")
            m_dd = gr.Dropdown(["Pipeline (best)"] + FINAL_NAMES, value="Pipeline (best)", label="Model")
            k_sl = gr.Slider(5, 20, 10, step=1, label="Top-K")
            l_sl = gr.Slider(0, 0.9, 0.0, step=0.1, label="Diversity (MMR λ)")
        b1 = gr.Button("Recommend", variant="primary")
        md1 = gr.Markdown()
        with gr.Row():
            h1 = gr.Dataframe(label="History (top-rated)"); r1 = gr.Dataframe(label="Recommendations")
        b1.click(gui_user, [u_dd, m_dd, k_sl, l_sl], [md1, h1, r1])
    with gr.Tab("เทียบโมเดล"):
        b2 = gr.Button("Compare"); c2 = gr.Dataframe(label="Top-K by model (✓ = liked in test)")
        b2.click(gui_compare, [u_dd, k_sl], c2)
    with gr.Tab("อธิบายค่าทำนาย"):
        mv_dd = gr.Dropdown(MOVIE_CHOICES, value=MOVIE_CHOICES[0], label="Movie")
        b3 = gr.Button("Explain"); md3 = gr.Markdown(); d3 = gr.Dataframe(label="SVD decomposition + all models")
        b3.click(gui_explain, [u_dd, mv_dd], [md3, d3])
    with gr.Tab("ผู้ใช้ใหม่ (Cold start)"):
        lk = gr.Dropdown(MOVIE_CHOICES, multiselect=True, label="เลือกเรื่องที่ชอบ (3–5 เรื่อง)")
        b4 = gr.Button("Recommend for new user"); md4 = gr.Markdown(); d4 = gr.Dataframe(label="Recommendations")
        b4.click(gui_newuser, [lk, k_sl], [md4, d4])

demo.launch(debug=False)